> 在线讲义：[第 11 章 · 时间与托管节点（lifecycle）](https://wikieden.github.io/ros2-from-scratch/lessons/11-time-lifecycle.html) · 本书《从零学 ROS 2》

# 第 11 章 · 时间与托管节点（lifecycle）

本章回答：仿真跑得比真实时间快或慢时，节点里的"现在几点"听谁的？回放 bag 时为什么要设 `use_sim_time`？一个驱动节点怎么做到"先配置好、确认没问题再开始干活、出错能退回去"？

配套讲义：仓库里 `ros2/notes/lessons/11-time-lifecycle.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：本机没有 rclpy。时间那部分用"仿真器每真实 1 秒推进几秒"来模拟 /clock；状态机部分照讲义第 2 步的转换表写成字典，生命周期发布者在不 Active 时 publish 不会发出去。

In [ ]:
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 第 1 步 仿真时间 | 定时器按 ROS 时间触发 | `timer_fire_wall_times(0.1, 0.1, 3.0)` | 真实 1.0、2.0、3.0 秒各触发一次 |
| 同上 | 仿真和真实一样快 | `timer_fire_wall_times(0.1, 1.0, 0.3)` | 0.1、0.2、0.3 秒 |
| 第 2 步 状态机 | configure → activate → … | `node.trigger(...)` | on_configure … on_shutdown 依次调用 |
| 第 2 步 托管发布者 | 不在 active 时不发 | `node.publish(...)` | 只有 #2 真的发出去 |
| 第 2 步 外部下令 | 不能跳步 | `trigger('activate')` 于 unconfigured | 报错 |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**常量 `TRANSITIONS`**

In [ ]:
TRANSITIONS = {  # (主状态, 转换) → 新的主状态；讲义第 2 步的表
    ("unconfigured", "configure"): "inactive",
    ("inactive", "activate"): "active",
    ("active", "deactivate"): "inactive",
    ("inactive", "cleanup"): "unconfigured",
    ("unconfigured", "shutdown"): "finalized",
    ("inactive", "shutdown"): "finalized",
    ("active", "shutdown"): "finalized",
}

**类 `ManagedNode`**：托管节点：状态只能由外部下令转换；不在 active 时 publish 不会真的发出去。

In [ ]:
class ManagedNode:
    """托管节点：状态只能由外部下令转换；不在 active 时 publish 不会真的发出去。"""

    def __init__(self, name: str):
        self.name = name
        self.state = "unconfigured"
        self.sent: list[str] = []
        self.log: list[str] = []

    def available_transitions(self) -> list[str]:
        """ros2 lifecycle list：当前状态下能做哪些转换。"""
        return [tr for (st, tr) in TRANSITIONS if st == self.state]

    def trigger(self, transition: str) -> str:
        """ros2 lifecycle set：调 on_<转换> 回调，成功就进入新状态。"""
        new = TRANSITIONS.get((self.state, transition))
        if new is None:
            raise ValueError(f"{self.state} 状态下不能 {transition}")
        self.log.append(f"on_{transition}: {self.state} → {new}")
        self.state = new
        return new

    def publish(self, msg: str) -> bool:
        if self.state != "active":
            return False
        self.sent.append(msg)
        return True

**函数 `timer_fire_wall_times`**：定时器按 ROS 时间（= /clock）触发：仿真速度是真实的 sim_speed 倍时，每次触发落在真实时间的几秒。

In [ ]:
def timer_fire_wall_times(period_sim: float, sim_speed: float, wall_seconds: float) -> list[float]:
    """定时器按 ROS 时间（= /clock）触发：仿真速度是真实的 sim_speed 倍时，每次触发落在真实时间的几秒。"""
    out, k = [], 1
    while k * period_sim / sim_speed <= wall_seconds + 1e-9:
        out.append(round(k * period_sim / sim_speed, 3))
        k += 1
    return out

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 · 仿真时间：定时器周期 0.1 秒（ROS 时间），仿真跑得比真实慢 10 倍")
print("  真实 3 秒内，定时器在真实时间这些时刻触发:", timer_fire_wall_times(0.1, 0.1, 3.0))
print("  仿真跑得和真实一样快时，真实 0.3 秒内:", timer_fire_wall_times(0.1, 1.0, 0.3))

In [ ]:
print("\n第二步 · 托管节点：外部按顺序下令")
node = ManagedNode("lc_talker")
print("  刚启动:", node.state, "；能做:", node.available_transitions())
print("  未激活时 publish →", node.publish("Lifecycle HelloWorld #1"))
for tr in ["configure", "activate"]:
    node.trigger(tr)
print("  configure、activate 之后:", node.state, "；publish →", node.publish("Lifecycle HelloWorld #2"))
for tr in ["deactivate", "cleanup", "shutdown"]:
    node.trigger(tr)
print("  回调按顺序被调用:")
for line in node.log:
    print("   ", line)
print("  真正发出去的消息:", node.sent)

In [ ]:
print("\n第三步 · 不按顺序下令会被拒绝：unconfigured 不能直接 activate")
try:
    ManagedNode("x").trigger("activate")
except ValueError as e:
    print("  ", e)

## 3. 代码里才看得到的

1. **只模拟了主状态**：真托管节点在两个主状态之间还有过渡状态（Configuring、Activating……），回调就在过渡状态里执行；这里把"进过渡状态 → 调回调 → 成功到新主状态"压成了一步。
2. **出错那条路（ErrorProcessing）没写**：讲义第 2 步的表里有，模拟里省掉了。
3. **仿真速度 0.1** 表示仿真里过 0.1 秒、真实要过 1 秒，所以 0.1 秒周期的定时器在真实里每秒才触发一次。

## 4. 自己改一改

- 把 `sim_speed` 改成 2.0（仿真比真实快一倍），看真实 1 秒里触发几次；
- 在 `TRANSITIONS` 里找一找：active 状态能直接 cleanup 吗？用 `available_transitions()` 验证。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **三种钟**：系统钟会跳，稳定钟不倒退，ROS 钟听 `/clock`。
- **仿真时间**：全体 `use_sim_time`，一个都不能漏。
- **四主态**：未配置、未激活、激活、已结束；配置、激活、停用、清理、关闭五种转换。
- **分工**：构造函数什么都不做；configure 准备好，activate 才开工；转换听外部指挥。